# RQ2: Retrieval Method and Reranking Impact

This notebook evaluates the impact of different retrieval methods (dense, sparse, hybrid) and the effect of reranking on retrieval quality.

**Research Question:** Does hybrid retrieval with reranking outperform individual retrieval methods?

**Evaluation Method:**
- Compare dense-only, sparse-only, and hybrid retrieval
- Measure impact of reranking on hybrid retrieval
- Metrics: recall, precision, MRR, coverage, answer quality

**Inputs:** Golden Standard, chunked and embedded documents
**Outputs:** RQ2 evaluation results saved to `data/artifacts/evaluation_results/rq2_results.json`

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Any

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.config import load_golden_standard, get_default_pipeline_config, BENCHMARK_QUESTION_IDS, ARTIFACTS_DIR
from rag.pipeline import build_index
from rag.evaluator import evaluate_pipeline_config

print("Imports successful. Project root:", project_root)

In [ ]:
# Smoke test configuration
SMOKE_TEST = True

if SMOKE_TEST:
    print("[SMOKE TEST MODE] Using subset: 1 company, 1 question")
    test_companies = ["AkerBP_2024"]
    test_question_ids = ["Q1"]
else:
    print("[FULL EVALUATION] Using all benchmark questions and companies")
    test_companies = None
    test_question_ids = BENCHMARK_QUESTION_IDS

In [ ]:
# Load Golden Standard
golden_standard = load_golden_standard()
print(f"Loaded Golden Standard: {len(golden_standard)} questions")

In [ ]:
# Define RQ2 configurations: compare retrieval methods
default_config = get_default_pipeline_config()

rq2_configs = {
    "RQ2_dense_only": {
        **default_config,
        "retrieval_method": "dense",
        "use_reranker": False,
    },
    "RQ2_sparse_only": {
        **default_config,
        "retrieval_method": "sparse",
        "use_reranker": False,
    },
    "RQ2_hybrid": {
        **default_config,
        "retrieval_method": "hybrid",
        "use_reranker": False,
    },
    "RQ2_hybrid_rerank": {
        **default_config,
        "retrieval_method": "hybrid",
        "use_reranker": True,
        "rerank_top_n": 10,
    },
}

print("RQ2 Configurations:")
for name, config in rq2_configs.items():
    print(f"  {name}:")
    print(f"    Retrieval: {config['retrieval_method']}")
    print(f"    Reranking: {'Yes' if config['use_reranker'] else 'No'}")

## Run RQ2 Evaluation

In [ ]:
# Run evaluation for each RQ2 configuration
rq2_results = {}

for config_name, config in rq2_configs.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {config_name}")
    print(f"{'='*60}")
    
    # Build index
    print(f"Building index with {config['retrieval_method']} retrieval...")
    vector_index, bm25_index, chunks = build_index(
        config=config,
        force_rebuild=False,
        companies=test_companies
    )
    print(f"Index built: {len(chunks)} chunks")
    
    # Evaluate
    print(f"Running evaluation...")
    result = evaluate_pipeline_config(
        config_name=config_name,
        retriever=None,
        config=config,
        golden_standard=golden_standard,
        question_ids=test_question_ids,
        companies=test_companies,
        skip_llm_judge=SMOKE_TEST,
        skip_bertscore=SMOKE_TEST
    )
    
    rq2_results[config_name] = result
    print(f"Evaluation complete for {config_name}")

## Retrieval Method Comparison

In [ ]:
# Build comparison table
metrics_data = []

for config_name, result in rq2_results.items():
    agg = result.get("aggregate", {})
    
    metrics_data.append({
        "Configuration": config_name,
        "Recall@5": agg.get("retrieval_recall_at_5", 0),
        "Recall@10": agg.get("retrieval_recall_at_10", 0),
        "Recall@15": agg.get("retrieval_recall_at_15", 0),
        "Precision@10": agg.get("retrieval_precision_at_10", 0),
        "MRR": agg.get("retrieval_mrr", 0),
        "Coverage": agg.get("retrieval_coverage", 0),
        "Token F1": agg.get("answer_token_f1", 0),
    })

comparison_df = pd.DataFrame(metrics_data)
print("\nRQ2 Retrieval Method Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
# Create grouped bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('RQ2: Retrieval Method Comparison', fontsize=14, fontweight='bold')

configs = list(rq2_results.keys())
x_pos = np.arange(len(configs))

# Recall@10
ax = axes[0, 0]
values = [rq2_results[c].get("aggregate", {}).get("retrieval_recall_at_10", 0) for c in configs]
ax.bar(x_pos, values, width=0.6)
ax.set_ylabel('Score')
ax.set_title('Recall@10')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('RQ2_', '') for c in configs], rotation=45)
ax.set_ylim([0, 1])

# Precision@10
ax = axes[0, 1]
values = [rq2_results[c].get("aggregate", {}).get("retrieval_precision_at_10", 0) for c in configs]
ax.bar(x_pos, values, width=0.6, color='steelblue')
ax.set_ylabel('Score')
ax.set_title('Precision@10')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('RQ2_', '') for c in configs], rotation=45)
ax.set_ylim([0, 1])

# MRR
ax = axes[1, 0]
values = [rq2_results[c].get("aggregate", {}).get("retrieval_mrr", 0) for c in configs]
ax.bar(x_pos, values, width=0.6, color='coral')
ax.set_ylabel('Score')
ax.set_title('Mean Reciprocal Rank')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('RQ2_', '') for c in configs], rotation=45)
ax.set_ylim([0, 1])

# Coverage
ax = axes[1, 1]
values = [rq2_results[c].get("aggregate", {}).get("retrieval_coverage", 0) for c in configs]
ax.bar(x_pos, values, width=0.6, color='lightgreen')
ax.set_ylabel('Score')
ax.set_title('Evidence Coverage')
ax.set_xticks(x_pos)
ax.set_xticklabels([c.replace('RQ2_', '') for c in configs], rotation=45)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'rq2_comparison.png', dpi=150, bbox_inches='tight')
print("Visualization saved to:", ARTIFACTS_DIR / 'rq2_comparison.png')
plt.show()

## Reranking Impact Analysis

In [ ]:
# Compare hybrid vs hybrid_rerank
if "RQ2_hybrid" in rq2_results and "RQ2_hybrid_rerank" in rq2_results:
    print("\n" + "="*60)
    print("Reranking Impact (Hybrid vs Hybrid+Rerank)")
    print("="*60)
    
    hybrid = rq2_results["RQ2_hybrid"].get("aggregate", {})
    hybrid_rerank = rq2_results["RQ2_hybrid_rerank"].get("aggregate", {})
    
    metrics_to_compare = [
        ("retrieval_recall_at_10", "Recall@10"),
        ("retrieval_precision_at_10", "Precision@10"),
        ("retrieval_mrr", "MRR"),
        ("retrieval_coverage", "Coverage"),
        ("answer_token_f1", "Token F1"),
    ]
    
    print(f"\n{'Metric':<20} {'Hybrid':<12} {'Hybrid+Rerank':<15} {'Delta':<10}")
    print("-" * 60)
    
    for metric_key, metric_name in metrics_to_compare:
        h_val = hybrid.get(metric_key, 0)
        hr_val = hybrid_rerank.get(metric_key, 0)
        delta = hr_val - h_val
        arrow = " UP" if delta > 0.01 else (" DOWN" if delta < -0.01 else " FLAT")
        print(f"{metric_name:<20} {h_val:<12.4f} {hr_val:<15.4f} {delta:+.4f}{arrow}")
else:
    print("Hybrid configurations not found in results")

In [ ]:
# Save RQ2 results
eval_results_dir = ARTIFACTS_DIR / "evaluation_results"
eval_results_dir.mkdir(parents=True, exist_ok=True)

output_path = eval_results_dir / "rq2_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(rq2_results, f, ensure_ascii=False, indent=2)

print(f"\nRQ2 results saved to: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")